# Part 4: U-Net Semantic Segmentation

把 Part 3 的影像 tile 与整数 mask 映射为逐像素类别，理解 encoder、decoder、skip connection、Dice 与 IoU。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`image [B,C,H,W] → encoder → bottleneck → decoder + skips → logits [B,K,H,W]`

In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd().resolve()
while not (ROOT / 'scripts').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
print('repository:', ROOT)

## 核心概念

- **Encoder–decoder**：编码器降低空间分辨率并增加通道；解码器恢复分辨率。
- **Skip connection**：拼接同尺度浅层细节，帮助恢复边界与小目标。
- **Logits 与 mask**：模型输出每类 logits；target 是 [B,H,W] 的 int64 类别索引。
- **Dice / IoU**：比 accuracy 更适合前景稀少的建筑、水体和道路分割。

## 本地真实训练结果

长训练由脚本执行；这里读取本机已经生成的指标和产物。GitHub 中不会包含 `outputs/`。

In [ ]:
output = ROOT / r'outputs/part4_unet'
metrics_path = output / 'metrics.json'
print('local output:', output)
print(json.dumps(json.loads(metrics_path.read_text(encoding='utf-8')), indent=2) if metrics_path.exists() else '尚未运行：请执行下方命令。')

## 可恢复的脚本训练

默认不在 Notebook 内启动长训练。将 `RUN_SHORT_TRAIN` 改为 `True` 可执行一次小规模 smoke run；正式命令请在终端执行。

In [ ]:
RUN_SHORT_TRAIN = False
command = r'''conda run --no-capture-output -n remote-sensing-dl python scripts/real_training/train_segmentation.py --model unet --data-root data/landcoverai --output outputs/part4_unet --epochs 1 --train-limit 32 --val-limit 16'''
print(command)
if RUN_SHORT_TRAIN:
    import subprocess
    subprocess.run(command, cwd=ROOT, shell=True, check=True)

## 结果解释

正式结果必须包含非空 checkpoint、完整 epoch 日志、验证指标与预测图；出现缺失时先检查数据配对、CUDA 与配置，而不要直接扩大模型。

## 练习

把阈值从 0.3 扫到 0.7，比较 Dice 最优阈值，并解释为什么测试集不能用于选阈值。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：skip feature 尺寸不一致；mask 被双线性插值；把 logits 先 argmax 再送入 loss；随机 tile 划分造成空间泄漏。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例